In [38]:
# importing libraries
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,balanced_accuracy_score, classification_report
from sklearn.dummy import DummyClassifier

In [5]:
#read in data
irrigation_data = pd.read_csv("C:\\Users\\donya\\Documents\\GSB 545\\GSB-545\\train.csv")
irrigation_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


## Data preparation and preprocessing

In [6]:
#target encode and split data
irrigation_data["Irrigation_Need"]=irrigation_data["Irrigation_Need"].map({
    "Low":0,
    "Medium":1,
    "High":2
})

print(irrigation_data.head())
print()
print(irrigation_data["Irrigation_Need"].value_counts())
print()
print(irrigation_data.isna().sum())

X=irrigation_data.drop(["id","Irrigation_Need"],axis=1)
y=irrigation_data["Irrigation_Need"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,stratify=y,random_state=321
)

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

In [12]:
X.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South


In [8]:
y.head()

0    0
1    0
2    0
3    1
4    0
Name: Irrigation_Need, dtype: int64

In [9]:
#identify numeric and categorical features
numeric_features=X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features=X_train.select_dtypes(include=["object"]).columns.tolist()

print("numeric columns:",numeric_features)
print("categorical columns:",categorical_features)

numeric columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


C:\Users\donya\AppData\Local\Temp\ipykernel_9604\2863354243.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features=X_train.select_dtypes(include=["object"]).columns.tolist()


In [19]:
#build preprocessor
categorical_transformer=Pipeline([
    ("onehot",OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor=ColumnTransformer([
    ("dummify",categorical_transformer,categorical_features)
], remainder = "passthrough").set_output(transform="pandas")

preprocessor.fit_transform(X).head()

,dummify__Soil_Type_Clay,dummify__Soil_Type_Loamy,dummify__Soil_Type_Sandy,dummify__Soil_Type_Silt,dummify__Crop_Type_Cotton,dummify__Crop_Type_Maize,dummify__Crop_Type_Potato,dummify__Crop_Type_Rice,dummify__Crop_Type_Sugarcane,dummify__Crop_Type_Wheat,...,remainder__Soil_Moisture,remainder__Organic_Carbon,remainder__Electrical_Conductivity,remainder__Temperature_C,remainder__Humidity,remainder__Rainfall_mm,remainder__Sunlight_Hours,remainder__Wind_Speed_kmh,remainder__Field_Area_hectare,remainder__Previous_Irrigation_mm
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,0.82,112.16
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,5.27,47.16
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,8.24,110.38
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,8.32,53.85
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,7.37,93.19


## GaussianNB model 

In [ ]:
#define model
nb_pipeline = Pipeline(
    [("preprocessing", preprocessor),
    ("nb", GaussianNB())]
)

In [28]:
#train and test model
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=321)

cv_scores = cross_val_score(nb_pipeline, X_train, y_train, cv=cv, scoring="balanced_accuracy", n_jobs=-1)
print(cv_scores)
print(cv_scores.mean())
print(cv_scores.std())

nb_pipeline.fit(X_train, y_train)

#evaluate final model performance on held out test set
y_pred = nb_pipeline.predict(X_test)
print("GaussianNB Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

[0.78100896 0.77554826 0.7732406  0.77737588 0.77864919]
0.7771645773858118
0.002647192472145199
GaussianNB Balanced Accuracy Score: 0.7775967334329107
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.80      0.83     73983
           1       0.70      0.76      0.73     47815
           2       0.53      0.77      0.63      4202

    accuracy                           0.78    126000
   macro avg       0.70      0.78      0.73    126000
weighted avg       0.80      0.78      0.79    126000



In [30]:
#predicted probabilities using model
probs = nb_pipeline.predict_proba(X_test)
probs_df = pd.DataFrame(probs, columns=[f"P(class={c})" for c in nb_pipeline.classes_])
print(probs_df.head())

   P(class=0)  P(class=1)    P(class=2)
0    0.999533    0.000467  8.051962e-27
1    0.999684    0.000316  4.836800e-29
2    0.398852    0.601148  2.622167e-07
3    0.746268    0.253732  1.085186e-07
4    0.063691    0.934467  1.841798e-03


In [34]:
#create baseline predictions using default rule (floor for balanced accuracy performance)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_dummy_pred = dummy.predict(X_test)

print("\nBaseline (default rule) performance:")
print(classification_report(y_test, y_dummy_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_dummy_pred))
print(y_dummy_pred)


Baseline (default rule) performance:
              precision    recall  f1-score   support

           0       0.59      1.00      0.74     73983
           1       0.00      0.00      0.00     47815
           2       0.00      0.00      0.00      4202

    accuracy                           0.59    126000
   macro avg       0.20      0.33      0.25    126000
weighted avg       0.34      0.59      0.43    126000

Balanced accuracy: 0.3333333333333333
[0 0 0 ... 0 0 0]


c:\Users\donya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\donya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\donya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

## Single class focus and threshold tuning

Since the "High" class is rare, I will be choosing this class to focus on. I will use recall as my metric.

In [51]:
def evaluate_thresholds(y_true, probs, classes, target_class=2, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 20)
    
    high_idx = list(classes).index(target_class)
    
    rows = []
    for t in thresholds:
        y_pred_new = []
        for i in range(len(y_true)):
            if probs[i, high_idx] >= t:
                y_pred_new.append(target_class)
            else:
                y_pred_new.append(classes[probs[i, :].argmax()])
        y_pred_new = np.array(y_pred_new)
        
        rows.append({
            "threshold": t,
            "recall": recall_score(y_true, y_pred_new, labels=[target_class], average="macro", zero_division=0),
        })
    return pd.DataFrame(rows)


results = evaluate_thresholds(y_test, probs, nb_pipeline.classes_, target_class=2)
results = results.sort_values(by="recall", ascending=False)
print(results)

    threshold    recall
0    0.100000  0.861257
1    0.142105  0.850309
2    0.184211  0.834841
3    0.226316  0.820562
4    0.268421  0.812470
5    0.310526  0.805569
6    0.352632  0.800333
7    0.394737  0.790338
8    0.436842  0.782485
9    0.478947  0.774869
10   0.521053  0.771537
11   0.563158  0.771537
12   0.605263  0.771537
13   0.647368  0.771537
14   0.689474  0.771537
15   0.731579  0.771537
16   0.773684  0.771537
17   0.815789  0.771537
18   0.857895  0.771537
19   0.900000  0.771537


In [ ]:
#generate new predictions with best threshold for maximizing recall
best_threshold = 0.1

high_idx = list(nb_pipeline.classes_).index(2)

y_pred_threshold = y_pred.copy()

y_pred_threshold[probs[:, high_idx] >= best_threshold] = 2

print(classification_report(y_test, y_pred_threshold))
print(y_pred_threshold)
print("GaussianNB Balanced Accuracy Score (New Threshold):", balanced_accuracy_score(y_test, y_pred_threshold))

              precision    recall  f1-score   support

           0       0.87      0.80      0.83     73983
           1       0.67      0.65      0.66     47815
           2       0.31      0.86      0.45      4202

    accuracy                           0.75    126000
   macro avg       0.62      0.77      0.65    126000
weighted avg       0.78      0.75      0.76    126000

[0 0 1 ... 1 0 0]
GaussianNB Balanced Accuracy Score (New Threshold): 0.7713568690659279


## Discussion

I used the "High" class (redefined as 2) as my selection because according to my previous Kaggle homework, this was the rarest class compared to "Low" and "Medium" irrigation needs. I used recall to find the optimal threshold because in my opinion, it is more important to correctly identify "High" irrigation need observations than to incorrectly identify an observation as "High" irrigation need when it is "Low" or "Medium". This threshold of 0.1 allowed us to reduce the number of false negatives ("High" irrigation need observations classified as "Low" or "Medium"). Specifically, the recall metric for "High" observations was able to increase substantially with this new threshold (0.86 vs. 0.77). However, there are definitely observable tradeoffs. Precision and f1-score decreased from 0.53 and 0.63 to 0.31 and 0.45, respectively. Optimizing the recall of the "High" observations also impacted the balanced accuracy score (decrease from 0.778 to 0.771).

Naive Bayes pros and cons compared to my other models

Pros
- Way faster code run time than tree models
- Easy to implement
- Less of a black box (easy to understand)

Cons
- Seems a bit too oversimplified
- Baseline performed worse than my tree models